In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import os
os.chdir("/home/zongchen/mmd_flow_cubature")
import matplotlib.cm as cm
import matplotlib

import sys
sys.path.append("/home/zongchen/mmd_flow_cubature")
from mmd_flow.kernels import gaussian_kernel
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")

from tqdm import tqdm

plt.rcParams['axes.grid'] = True
plt.rcParams['font.family'] = 'DeJavu Serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.labelsize'] = 18
plt.rc('text', usetex=True)
plt.rc('text.latex', preamble=r'\usepackage{amsmath, amsfonts}')

# plt.rc('font', family='Arial', size=12)
plt.rc('axes', titlesize=16, labelsize=18, grid=True)
plt.rc('lines', linewidth=2)
plt.rc('legend', fontsize=18, frameon=False)
plt.rc('xtick', labelsize=14, direction='in')
plt.rc('ytick', labelsize=14, direction='in')
plt.rc('figure', figsize=(6, 4), dpi=100)

In [ ]:
from mmd_flow.distributions import Cross
from mmd_flow.kernels import gaussian_kernel

k = 2
w = 0.2
h = 1.0
skip = 1.5
kernel = gaussian_kernel(0.5)
distribution = Cross(kernel=kernel, w=w, h=h, k=k, skip=skip)

rng_key = jax.random.PRNGKey(1)
iid_samples = distribution.sample(20, rng_key)

resolution = 100
h_prime = 0.75
x_lower, x_upper = -h_prime - skip * (k-1), h_prime
y_lower, y_upper = -h_prime, h_prime + skip * (k-1)
x = np.linspace(x_lower, x_upper, resolution)
y = np.linspace(y_lower, y_upper, resolution)
X, Y = np.meshgrid(x, y)

pdf = distribution.pdf(jnp.array([X.flatten(), Y.flatten()]).T)
pdf = pdf.reshape(X.shape)

# Load MMD flow samples and use the last iterate as stationary points
sample_path = "/home/zongchen/mmd_flow_cubature/results/mmd_flow/cross_dataset/Gaussian_kernel/__step_size_0.3__dim_2__bandwidth_0.2__step_num_5000__particle_num_20__inject_noise_scale_1.0__inject_noise_schedule_t^-0.5__seed_0__complete/mmd_flow_samples.npy"
mmd_flow_samples = jnp.load(sample_path)
if mmd_flow_samples.ndim == 3:
    stationary_points = mmd_flow_samples[-1]
elif mmd_flow_samples.ndim == 2:
    stationary_points = mmd_flow_samples
else:
    raise ValueError(f"Unexpected sample shape: {mmd_flow_samples.shape}")

print("Loaded mmd_flow_samples shape:", mmd_flow_samples.shape)
print("Using stationary_points shape:", stationary_points.shape)

In [ ]:
import matplotlib.colors as mcolors
# 204, 230, 225
custom_color = (204/255, 230/255, 225/255)
cmap = mcolors.ListedColormap([custom_color])
cmap.set_bad('white')  # NaN will be white

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for i in range(2):
    axs[i].imshow(pdf, extent=(x_lower, x_upper, y_lower, y_upper), vmin=0, vmax=1, cmap=cmap)
    axs[i].grid(False)
    axs[i].set_xticks([])
    axs[i].set_yticks([])
    axs[i].set_xlabel('')
    axs[i].set_ylabel('')

axs[0].scatter(iid_samples[:, 0], iid_samples[:, 1], s=20, c='black', alpha=1.0)
axs[0].set_title("IID")

axs[1].scatter(stationary_points[:, 0], stationary_points[:, 1], s=20, c='salmon', alpha=1.0)
axs[1].set_title("Ours")

save_path = os.path.expanduser('~/mini_mmd/figures/illustration_sampling.pdf')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
plt.savefig(save_path, bbox_inches='tight')
plt.show()